# Notebook 3 of 5 · Claude API Classification
### The policy-native *challenger* control

> **Why an LLM challenger?** The baseline keys on tokens; it cannot reason about *coded*
> hate or *veiled* threats with no explicit keyword. A frontier model can — and, crucially,
> it can judge content against **Anthropic Usage Policy clauses** rather than abstract
> dataset labels, so its output speaks the language a safety team governs in. Logic:
> [llm_client.py](../src/llm_client.py); model: `claude-opus-4-8`.

> **No API key?** This notebook degrades gracefully: with no `ANTHROPIC_API_KEY` it uses a
> deterministic offline heuristic *explicitly labelled as not a live-model verdict*, so the
> pipeline stays reproducible. Live results require the key.

In [ ]:
# --- repo bootstrap: make `from src import ...` work from notebooks/ ---
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.titleweight": "bold", "font.size": 10})

import src
from src import data_loader as dl
from src.data_loader import LABELS, ANTHROPIC_POLICY_MAP, SEVERITY_ORDER
print("src loaded. LABELS:", LABELS)

In [ ]:
from src import ml_baseline as mlb
from src import safety_metrics as sm
from src.llm_client import ToxicityClassifier, assessments_to_frame, SYSTEM_PROMPT

clf = ToxicityClassifier()
LIVE = clf.is_available()
print("Claude API available:", LIVE, "| model:", clf.model)
print("Mode:", "LIVE Claude calls" if LIVE else "OFFLINE heuristic (labelled, reproducible)")

## 1 · The policy-native system prompt

> The system prompt **is part of the control surface** and is built programmatically from
> the canonical policy map (`src.llm_client.build_system_prompt`) so prompt and mapping can
> never drift. Note the explicit **fairness rule**: neutral identity mentions must not be
> flagged.

In [ ]:
print(SYSTEM_PROMPT)

## 2 · Evaluation sample

> Full-corpus LLM evaluation is out of scope on cost grounds (and we say so — silent
> sampling reads as full coverage). We draw a **balanced sample**: all flagged comments are
> rare, so we oversample them to get signal on the categories that matter, then benchmark
> the LLM and the baseline on the *same* sample.

In [ ]:
df = dl.load_jigsaw()
rng = np.random.RandomState(42)
any_label = df[LABELS].sum(axis=1) > 0
SAMPLE_N = 120          # small, balanced; raise if running live with budget
n_pos = SAMPLE_N // 2
pos_idx = rng.choice(df.index[any_label], size=min(n_pos, int(any_label.sum())), replace=False)
neg_idx = rng.choice(df.index[~any_label], size=SAMPLE_N - len(pos_idx), replace=False)
sample = df.loc[np.concatenate([pos_idx, neg_idx])].sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Eval sample: {len(sample)} comments ({int(sample[LABELS].sum(axis=1).gt(0).sum())} flagged)")

## 3 · Run the Claude (or offline) classifier

In [ ]:
# classify_batch falls back per-comment to the offline heuristic on any error,
# so this runs to completion with or without an API key.
assessments = clf.classify_batch(sample["comment_text"].tolist())
llm_pred = assessments_to_frame(assessments)
llm_pred.to_csv(os.path.join(dl.outputs_dir(), "llm_predictions_sample.csv"), index=False)
print("Classified", len(assessments), "comments. Example policy-native verdicts:")
for i in range(3):
    a = assessments[i]
    print("-"*80)
    print("text:", dl.redact(sample.at[i, "comment_text"]))
    print("flags:", {l: getattr(a, l) for l in LABELS if getattr(a, l)} or "clean")
    print("policy_clauses:", a.policy_clauses)
    print("rationale:", a.rationale)

## 4 · Challenger vs champion — LLM vs baseline ML

> The governance question is not "is the LLM good?" but "**does the challenger beat the
> champion enough to justify its cost?**" We score both on the identical sample.

In [ ]:
# baseline on the same sample (load saved model, else train quickly)
try:
    base = mlb.load_model()
except Exception:
    base, _ = mlb.train_baseline(dl.load_jigsaw(nrows=40000))
_, ml_pred = mlb.predict(base, sample["comment_text"])
y_true = sample[LABELS].values

llm_sc = sm.per_category_metrics(y_true, llm_pred.values).set_index("category")
ml_sc  = sm.per_category_metrics(y_true, ml_pred.values).set_index("category")
compare = pd.DataFrame({
    "severity":   [ANTHROPIC_POLICY_MAP[l]["severity_tier"] for l in LABELS],
    "ml_recall":  ml_sc.loc[LABELS, "recall"].values,
    "llm_recall": llm_sc.loc[LABELS, "recall"].values,
    "ml_f1":      ml_sc.loc[LABELS, "f1"].values,
    "llm_f1":     llm_sc.loc[LABELS, "f1"].values,
}, index=LABELS).round(3)
compare

In [ ]:
x = np.arange(len(LABELS))
fig, ax = plt.subplots(figsize=(11, 4.3))
ax.bar(x-0.2, compare["ml_f1"], 0.4, label="ML baseline (champion)", color="#4C72B0")
ax.bar(x+0.2, compare["llm_f1"], 0.4, label=f"Claude {'(LIVE)' if LIVE else '(offline stand-in)'}", color="#C44E52")
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=30, ha="right")
ax.set_ylabel("F1 on eval sample"); ax.set_ylim(0, 1)
ax.set_title("Challenger vs champion: per-category F1")
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(dl.results_dir(), "llm_vs_ml_comparison.png"), bbox_inches="tight"); plt.show()
if not LIVE:
    print("NOTE: offline mode — these LLM numbers are a heuristic stand-in, NOT a Claude verdict.")

## 5 · Cost / latency / opacity trade-off

> A governed decision needs the trade made explicit, not assumed.

| Dimension | ML baseline (champion) | Claude challenger |
|---|---|---|
| Cost / 1k comments | ~free (CPU) | API token cost |
| Latency | milliseconds | ~seconds per call |
| Interpretability | high (token weights) | rationale text, but opaque internals |
| Coded/veiled harm | weak (keyword-bound) | strong (reasons over meaning) |
| Reliability risk | none | schema/hallucination risk → [HALLUCINATION_MONITORING.md](../docs/HALLUCINATION_MONITORING.md) |

**Governance conclusion.** The challenger earns its place where the champion is structurally
blind — coded hate and veiled threats — which is exactly what the **red-team battery (NB4)**
is built to prove. For high-volume, clearly-keyworded content the cheap champion is the right
control. A real system would **route**: cheap model first, escalate ambiguous cases to the LLM.

## 6 · Summary
- Built a **policy-native** Claude classifier with structured output and a fairness rule.
- Benchmarked challenger vs champion on a balanced sample; saved LLM predictions to `outputs/`.
- Framed the deploy decision as a **routing** trade-off, not a winner-take-all.

*Next → **Notebook 4: Red Teaming** — does either control survive an adversary?*